# 🎯 Tutorial 03: Prototypical Networks - Few-Shot Classification

## Tu Primer Algoritmo Real de Meta-Learning

En este tutorial aprenderás:

- 📐 Qué son Prototypical Networks y cómo funcionan
- 🔢 Aprendizaje basado en métricas (distance-based learning)
- 💻 Implementación completa de Few-Shot Classification
- 🎨 Visualización de embeddings y prototipos

---

## 📖 Parte 1: Teoría

### ¿Qué son Prototypical Networks?

**Prototypical Networks** (Snell et al., 2017) son una forma elegante de hacer Few-Shot Learning basándose en la idea de que:

> "Cada clase puede ser representada por su prototipo (centroide) en un espacio de embeddings."

### Intuición:

Imagina que quieres clasificar animales (perros, gatos, pájaros) pero solo tienes 3 fotos de cada uno:

1. **Embedding**: Convierte cada imagen en un vector numérico (embedding)
2. **Prototipo**: Calcula el promedio de los embeddings de cada clase
3. **Clasificación**: Asigna una nueva imagen a la clase cuyo prototipo esté más cerca

### Matemáticamente:

Para una clase $c$ con ejemplos de soporte $S_c = \{(x_i, y_i)\}$:

1. **Prototipo de la clase $c$**:
   $$p_c = \frac{1}{|S_c|} \sum_{(x_i, y_i) \in S_c} f_\theta(x_i)$$
   
   donde $f_\theta$ es la red de embeddings.

2. **Probabilidad de clasificación** (usando distancia euclidiana):
   $$P(y = c | x) = \frac{\exp(-d(f_\theta(x), p_c))}{\sum_{c'} \exp(-d(f_\theta(x), p_{c'}))}$$
   
   donde $d$ es la distancia euclidiana.

### Ventajas:

- ✅ Simple y elegante
- ✅ No requiere fine-tuning para nuevas clases
- ✅ Funciona muy bien en Few-Shot scenarios
- ✅ Interpretable (puedes visualizar los embeddings)


---

## 🛠️ Parte 2: Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import sys
sys.path.append('..')

from utils.test_utils import (
    print_success, print_hint, HintSystem, 
    run_test, check_model_output_shape
)
from utils.data_utils import create_classification_task, set_seed
from utils.visualization import plot_embeddings

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Dispositivo: {device}")
print("✅ Setup completo!")

---

## 💻 Parte 3: Ejercicio 1 - Red de Embeddings

Primero necesitamos una red que convierta las imágenes en embeddings.

**Tu tarea**: Completa la red de embeddings (CNN simple).

In [ ]:
class EmbeddingNetwork(nn.Module):
    """
    Red convolucional para generar embeddings.
    
    Input: Imágenes [batch, channels, height, width]
    Output: Embeddings [batch, embedding_dim]
    """
    
    def __init__(self, input_channels=1, embedding_dim=64):
        super(EmbeddingNetwork, self).__init__()
        
        # TODO: Define la arquitectura CNN
        # Arquitectura sugerida:
        # Conv1: input_channels -> 64, kernel 3x3, padding 1
        # Conv2: 64 -> 64, kernel 3x3, padding 1
        # Conv3: 64 -> 64, kernel 3x3, padding 1
        # Conv4: 64 -> embedding_dim, kernel 3x3, padding 1
        
        self.conv1 = None  # TODO: nn.Conv2d(...)
        self.conv2 = None  # TODO: nn.Conv2d(...)
        self.conv3 = None  # TODO: nn.Conv2d(...)
        self.conv4 = None  # TODO: nn.Conv2d(...)
        
        # Batch normalization (opcional pero recomendado)
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(64)
    
    def forward(self, x):
        """
        Args:
            x: [batch, channels, height, width]
        
        Returns:
            embeddings: [batch, embedding_dim]
        """
        # TODO: Implementa el forward pass
        # Usa: Conv -> BatchNorm -> ReLU -> MaxPool para cada bloque
        # Al final: Global Average Pooling (usa F.adaptive_avg_pool2d)
        
        pass  # TODO: Reemplaza con tu código


# Sistema de pistas
hints_embedding = HintSystem([
    "Conv2d toma: in_channels, out_channels, kernel_size, padding.",
    "Después de cada conv: aplica batch norm, ReLU, y max pooling con kernel 2x2.",
    "Al final usa F.adaptive_avg_pool2d(x, (1, 1)) para hacer global average pooling.",
    "Estructura: x = pool(relu(bn(conv(x)))); ... ; x = adaptive_avg_pool2d(x, (1,1)); x = x.view(x.size(0), -1)"
])

In [ ]:
# Para ver pistas
hints_embedding.show_hint()

In [ ]:
# ✅ TEST 1: Verificar construcción

def test_embedding_network():
    model = EmbeddingNetwork(input_channels=1, embedding_dim=64)
    
    # Verificar que las capas existen
    assert model.conv1 is not None, "conv1 no debe ser None"
    assert model.conv4 is not None, "conv4 no debe ser None"
    
    print_success("✅ Red de embeddings construida correctamente!")

run_test(test_embedding_network, "Test de Construcción")

In [ ]:
# ✅ TEST 2: Verificar forward pass

def test_embedding_forward():
    model = EmbeddingNetwork(input_channels=1, embedding_dim=64)
    x = torch.randn(5, 1, 28, 28)  # Batch de 5 imágenes 28x28
    
    embeddings = model(x)
    
    # Verificar shape
    assert embeddings.shape == (5, 64), f"Output debe ser (5, 64), pero es {embeddings.shape}"
    
    # Verificar que no hay NaN
    assert not torch.isnan(embeddings).any(), "Embeddings contienen NaN"
    
    print_success("✅ Forward pass funciona correctamente!")

run_test(test_embedding_forward, "Test de Forward Pass")

---

## 💻 Parte 4: Ejercicio 2 - Calcular Prototipos

El corazón de Prototypical Networks: calcular los prototipos de cada clase.

**Tu tarea**: Implementa la función que calcula prototipos.

In [ ]:
def compute_prototypes(embeddings, labels, n_way):
    """
    Calcula el prototipo (centroide) de cada clase.
    
    Args:
        embeddings: [n_samples, embedding_dim] - Embeddings del support set
        labels: [n_samples] - Etiquetas (0, 1, ..., n_way-1)
        n_way: Número de clases
    
    Returns:
        prototypes: [n_way, embedding_dim] - Un prototipo por clase
    """
    # TODO: Implementa el cálculo de prototipos
    # Para cada clase c:
    #   1. Encuentra todos los embeddings de esa clase
    #   2. Calcula el promedio (mean) de esos embeddings
    #   3. Ese promedio es el prototipo de la clase
    
    pass  # TODO: Reemplaza con tu código


# Sistema de pistas
hints_prototypes = HintSystem([
    "Usa un loop sobre las clases: for c in range(n_way).",
    "Para filtrar embeddings de clase c: mask = (labels == c); class_embeddings = embeddings[mask].",
    "El prototipo es el promedio: prototype_c = class_embeddings.mean(dim=0).",
    "Finalmente, apila todos los prototipos: prototypes = torch.stack([p0, p1, ..., p_n])."
])

In [ ]:
# Para ver pistas
hints_prototypes.show_hint()

In [ ]:
# ✅ TEST 3: Verificar cálculo de prototipos

def test_prototypes():
    # Crear datos de prueba
    embeddings = torch.randn(15, 64)  # 15 ejemplos, 64 dim
    labels = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4])  # 5 clases, 3 ejemplos cada una
    
    prototypes = compute_prototypes(embeddings, labels, n_way=5)
    
    # Verificar shape
    assert prototypes.shape == (5, 64), f"Prototypes debe ser (5, 64), pero es {prototypes.shape}"",
    
    # Verificar que el prototipo de clase 0 es el promedio de los primeros 3 embeddings
    expected_proto_0 = embeddings[0:3].mean(dim=0)
    assert torch.allclose(prototypes[0], expected_proto_0, atol=1e-5), "Prototipo de clase 0 incorrecto"
    
    print_success("✅ Cálculo de prototipos correcto!")

run_test(test_prototypes, "Test de Prototipos")

---

## 💻 Parte 5: Ejercicio 3 - Distancias Euclidianas

Necesitamos calcular distancias entre embeddings y prototipos.

**Tu tarea**: Implementa la función de distancias.

In [ ]:
def euclidean_distances(queries, prototypes):
    """
    Calcula distancias euclidianas entre queries y prototipos.
    
    Args:
        queries: [n_queries, embedding_dim]
        prototypes: [n_way, embedding_dim]
    
    Returns:
        distances: [n_queries, n_way] - Distancia de cada query a cada prototipo
    """
    # TODO: Calcula distancias euclidianas
    # Distancia euclidiana: d(a, b) = sqrt(sum((a - b)^2))
    # O usa: torch.cdist(queries, prototypes) - función built-in!
    
    pass  # TODO: Reemplaza con tu código


# Sistema de pistas
hints_distances = HintSystem([
    "PyTorch tiene una función built-in: torch.cdist(a, b) calcula distancias por pares.",
    "torch.cdist(queries, prototypes) devuelve matriz [n_queries, n_way] con todas las distancias.",
    "Si quieres implementarlo manualmente: expande dimensiones y usa (queries - prototypes).pow(2).sum(-1).sqrt().",
    "Solución simple: return torch.cdist(queries, prototypes)"
])

In [ ]:
# Para ver pistas
hints_distances.show_hint()

In [ ]:
# ✅ TEST 4: Verificar distancias

def test_distances():
    queries = torch.tensor([[1.0, 0.0], [0.0, 1.0]])  # 2 queries
    prototypes = torch.tensor([[0.0, 0.0], [1.0, 1.0]])  # 2 prototipos
    
    dists = euclidean_distances(queries, prototypes)
    
    # Verificar shape
    assert dists.shape == (2, 2), f"Distances debe ser (2, 2), pero es {dists.shape}"
    
    # Verificar distancias conocidas
    # query[0] = [1,0], proto[0] = [0,0] -> dist = 1.0
    assert torch.allclose(dists[0, 0], torch.tensor(1.0), atol=1e-5), "Distancia [0,0] incorrecta"
    
    print_success("✅ Cálculo de distancias correcto!")

run_test(test_distances, "Test de Distancias")

---

## 💻 Parte 6: Ejercicio 4 - Prototypical Network Completa

Ahora juntamos todo en una clase completa.

**Tu tarea**: Completa el método `classify`.

In [ ]:
class PrototypicalNetwork(nn.Module):
    """
    Prototypical Network completa para Few-Shot Classification.
    """
    
    def __init__(self, input_channels=1, embedding_dim=64):
        super(PrototypicalNetwork, self).__init__()
        self.encoder = EmbeddingNetwork(input_channels, embedding_dim)
    
    def forward(self, support_x, support_y, query_x, n_way):
        """
        Forward pass completo.
        
        Args:
            support_x: [n_support, channels, H, W] - Support set images
            support_y: [n_support] - Support labels
            query_x: [n_query, channels, H, W] - Query set images
            n_way: Número de clases
        
        Returns:
            logits: [n_query, n_way] - Logits de clasificación
        """
        # Codificar support y query
        support_embeddings = self.encoder(support_x)
        query_embeddings = self.encoder(query_x)
        
        # Calcular prototipos
        prototypes = compute_prototypes(support_embeddings, support_y, n_way)
        
        # Calcular distancias
        distances = euclidean_distances(query_embeddings, prototypes)
        
        # Convertir distancias a logits (negativo de distancias)
        logits = -distances
        
        return logits
    
    def classify(self, support_x, support_y, query_x, n_way):
        """
        Clasifica queries dados los ejemplos de soporte.
        
        Args:
            support_x, support_y, query_x, n_way: Igual que forward
        
        Returns:
            predictions: [n_query] - Clases predichas
            probabilities: [n_query, n_way] - Probabilidades
        """
        # TODO: Implementa la clasificación
        # 1. Obtén logits usando forward()
        # 2. Convierte logits a probabilidades usando softmax
        # 3. Obtén predicciones usando argmax
        
        pass  # TODO: Reemplaza con tu código


# Sistema de pistas
hints_classify = HintSystem([
    "Usa self.forward() para obtener los logits.",
    "Convierte logits a probabilidades: probs = F.softmax(logits, dim=1).",
    "Obtén predicciones: predictions = logits.argmax(dim=1).",
    "Código: logits = self.forward(...); probs = F.softmax(logits, dim=1); preds = logits.argmax(dim=1); return preds, probs"
])

In [ ]:
# Para ver pistas
hints_classify.show_hint()

In [ ]:
# ✅ TEST 5: Verificar Prototypical Network

def test_prototypical_network():
    model = PrototypicalNetwork(input_channels=1, embedding_dim=64)
    
    # Crear tarea sintética
    task = create_classification_task(n_way=5, k_shot=3, q_query=10, img_size=28, n_channels=1)
    
    # Forward pass
    logits = model(
        task['x_support'], 
        task['y_support'],
        task['x_query'],
        n_way=5
    )
    
    # Verificar shape
    assert logits.shape == (50, 5), f"Logits debe ser (50, 5), pero es {logits.shape}"  # 5 clases * 10 queries
    
    print_success("✅ Prototypical Network funciona correctamente!")

run_test(test_prototypical_network, "Test de Prototypical Network")

---

## 📊 Parte 7: Entrenamiento

Vamos a entrenar el modelo en múltiples tareas.

In [ ]:
def train_prototypical(model, n_episodes=1000, n_way=5, k_shot=5, q_query=15, lr=0.001):
    """
    Entrena Prototypical Network por episodios.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    model.to(device)
    
    losses = []
    accuracies = []
    
    print(f"🚀 Entrenando por {n_episodes} episodios...\n")
    
    for episode in range(n_episodes):
        # Sample a task
        task = create_classification_task(n_way=n_way, k_shot=k_shot, q_query=q_query)
        
        # Move to device
        support_x = task['x_support'].to(device)
        support_y = task['y_support'].to(device)
        query_x = task['x_query'].to(device)
        query_y = task['y_query'].to(device)
        
        # Forward
        model.train()
        logits = model(support_x, support_y, query_x, n_way=n_way)
        loss = criterion(logits, query_y)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
        acc = (logits.argmax(dim=1) == query_y).float().mean().item()
        
        losses.append(loss.item())
        accuracies.append(acc)
        
        if (episode + 1) % 100 == 0:
            avg_loss = np.mean(losses[-100:])
            avg_acc = np.mean(accuracies[-100:])
            print(f"Episodio {episode+1}/{n_episodes} - Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}")
    
    return losses, accuracies


# Crear y entrenar modelo
protonet = PrototypicalNetwork(input_channels=1, embedding_dim=64)

losses, accs = train_prototypical(
    protonet, 
    n_episodes=500,
    n_way=5, 
    k_shot=5, 
    q_query=15,
    lr=0.001
)

print("\n✅ Entrenamiento completado!")

In [ ]:
# Visualizar curvas de aprendizaje
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(losses, alpha=0.3, color='blue')
ax1.plot(np.convolve(losses, np.ones(50)/50, mode='valid'), color='blue', linewidth=2)
ax1.set_xlabel('Episodio')
ax1.set_ylabel('Loss')
ax1.set_title('Curva de Loss')
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(accs, alpha=0.3, color='green')
ax2.plot(np.convolve(accs, np.ones(50)/50, mode='valid'), color='green', linewidth=2)
ax2.set_xlabel('Episodio')
ax2.set_ylabel('Accuracy')
ax2.set_title('Curva de Accuracy')
ax2.axhline(y=1.0/5, color='r', linestyle='--', label='Random (20%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Accuracy final promedio: {np.mean(accs[-100:]):.2%}")

---

## 🎯 Parte 8: Evaluación en Nuevas Tareas

Probemos el modelo en tareas completamente nuevas.

In [ ]:
def evaluate_prototypical(model, n_test_episodes=100, n_way=5, k_shot=5, q_query=15):
    """
    Evalúa el modelo en episodios de test.
    """
    model.eval()
    accuracies = []
    
    with torch.no_grad():
        for _ in range(n_test_episodes):
            task = create_classification_task(n_way=n_way, k_shot=k_shot, q_query=q_query)
            
            logits = model(
                task['x_support'].to(device),
                task['y_support'].to(device),
                task['x_query'].to(device),
                n_way=n_way
            )
            
            acc = (logits.argmax(dim=1) == task['y_query'].to(device)).float().mean().item()
            accuracies.append(acc)
    
    return accuracies


print("🧪 Evaluando en diferentes escenarios...\n")

scenarios = [
    (5, 1),   # 5-way 1-shot
    (5, 5),   # 5-way 5-shot
    (10, 1),  # 10-way 1-shot
    (10, 5),  # 10-way 5-shot
]

for n_way, k_shot in scenarios:
    accs = evaluate_prototypical(protonet, n_test_episodes=100, n_way=n_way, k_shot=k_shot)
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    
    print(f"{n_way}-way {k_shot}-shot: {mean_acc:.2%} ± {std_acc:.2%}")

print("\n✅ Evaluación completada!")

---

## 🎓 Resumen y Conclusiones

### ✅ Lo que aprendiste:

1. **Prototypical Networks** usan métricas (distancias) para clasificación
2. Cada clase se representa por su **prototipo** (promedio de embeddings)
3. La clasificación es simplemente **encontrar el prototipo más cercano**
4. No se requiere **fine-tuning** para nuevas clases

### 🔍 Ventajas de Prototypical Networks:

- ✅ Simple e interpretable
- ✅ Eficiente (no requiere re-entrenamiento)
- ✅ Funciona bien con pocos ejemplos
- ✅ Fácil de extender a nuevas clases

### ⚖️ Limitaciones:

- ⚠️ Asume que el espacio de embeddings es euclidiano
- ⚠️ No aprende explícitamente el proceso de adaptación
- ⚠️ Puede ser sub-óptimo para tareas complejas

### 🚀 Próximo Tutorial:

En el **Tutorial 04 (MAML)** veremos un enfoque diferente que aprende explícitamente cómo adaptar rápidamente los pesos del modelo usando gradientes de segundo orden.

---

## 🎉 ¡Felicidades!

Has implementado tu primer algoritmo real de Meta-Learning! Prototypical Networks son la base para entender métodos más avanzados.
